In [10]:
import nflreadpy as nfl
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [45]:
schedules_polars = nfl.load_schedules(seasons=True)
schedules = schedules_polars.to_pandas()
# schedules = schedules.dropna(subset=["spread_line", "result"])

# schedules.to_csv('schedules.csv', index=False)

In [24]:
#Split into train and test sets

#TODO: FIGURE OUT HOW TO CONVERT ANY KEY COLUMNS I WANT INTO NUM, STRING NOT ALLOWED FOR SCALER AND IN GENERAL
X = schedules[['spread_line', 'week']]
y = schedules['result'] > 0 #Convert result from point dif to binary outcome 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Scale features (put everything on scale 0-1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = LogisticRegression(max_iter=10000)
model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [26]:
#Do prediction on test set
y_pred = model.predict(X_test_scaled)

In [30]:
# Test model accuracy results
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Loss", "Win"]))

Model Accuracy: 0.6641

Classification Report:
              precision    recall  f1-score   support

        Loss       0.65      0.52      0.58       643
         Win       0.67      0.78      0.72       813

    accuracy                           0.66      1456
   macro avg       0.66      0.65      0.65      1456
weighted avg       0.66      0.66      0.66      1456



In [57]:
week1 =schedules[(schedules['week'] ==1) & (schedules['season'] == 2026)]
week1 = week1.reset_index(drop=True)
week1_scaled = scaler.transform(week1[['spread_line', 'week']])
week1_probs = model.predict_proba(week1_scaled)[:, 1]
print(week1_probs)


[0.59007215 0.60718269 0.37980078 0.60718269 0.71789325 0.43125576
 0.36317378 0.75909051 0.60718269 0.53757881 0.78417133 0.59007215
 0.53757881 0.67268869 0.37980078 0.57274074]


In [60]:
for index, row in week1.iterrows():
    print(f"Game: {row['away_team']} @ {row['home_team']}, Predicted Win Probability for Home Team: {week1_probs[index]:.4f}")

Game: NE @ SEA, Predicted Win Probability for Home Team: 0.5901
Game: SF @ LA, Predicted Win Probability for Home Team: 0.6072
Game: CHI @ CAR, Predicted Win Probability for Home Team: 0.3798
Game: TB @ CIN, Predicted Win Probability for Home Team: 0.6072
Game: NO @ DET, Predicted Win Probability for Home Team: 0.7179
Game: BUF @ HOU, Predicted Win Probability for Home Team: 0.4313
Game: BAL @ IND, Predicted Win Probability for Home Team: 0.3632
Game: CLE @ JAX, Predicted Win Probability for Home Team: 0.7591
Game: ATL @ PIT, Predicted Win Probability for Home Team: 0.6072
Game: NYJ @ TEN, Predicted Win Probability for Home Team: 0.5376
Game: ARI @ LAC, Predicted Win Probability for Home Team: 0.7842
Game: MIA @ LV, Predicted Win Probability for Home Team: 0.5901
Game: GB @ MIN, Predicted Win Probability for Home Team: 0.5376
Game: WAS @ PHI, Predicted Win Probability for Home Team: 0.6727
Game: DAL @ NYG, Predicted Win Probability for Home Team: 0.3798
Game: DEN @ KC, Predicted Win Pr

In [ ]:
#Account for points based on spread. 1pt for favorite, 2 point for underdog. 3 points for 7 pt underdog
#add cols for homeIsFavorite (bool), 7ptUnderdog (bool), evHome (float), evAway (float), bestEV ("Home or Away")
week1['homeIsFavorite'] = week1['spread_line'] < 0
week1['7ptUnderdog'] = abs(week1['spread_line']) >= 7
for index, row in week1.iterrows():
    
print(week1[['away_team', 'home_team', 'spread_line', '7ptUnderdog', 'evHome', 'evAway', 'bestEV']])

   away_team home_team  spread_line  7ptUnderdog    evHome    evAway bestEV
0         NE       SEA          3.0        False  1.180144  0.409928   Home
1         SF        LA          3.5        False  1.214365  0.392817   Home
2        CHI       CAR         -3.0        False  0.379801  1.240398   Away
3         TB       CIN          3.5        False  1.214365  0.392817   Home
4         NO       DET          7.0         True  2.153680  0.000000   Home
5        BUF       HOU         -1.5        False  0.431256  1.137488   Away
6        BAL       IND         -3.5        False  0.363174  1.273652   Away
7        CLE       JAX          8.5         True  2.277272  0.000000   Home
8        ATL       PIT          3.5        False  1.214365  0.392817   Home
9        NYJ       TEN          1.5        False  1.075158  0.462421   Home
10       ARI       LAC          9.5         True  2.352514  0.000000   Home
11       MIA        LV          3.0        False  1.180144  0.409928   Home
12        GB